# MyBabyAI Cloud Trainer (Optimized)

This notebook provides a professional training environment for the **CodeMind** model with advanced monitoring, multi-dataset support (English/Turkish), and VRAM optimization.

In [ ]:
# 1. Install dependencies
!pip install torch torchvision transformers accelerate peft bitsandbytes sentencepiece tokenizers safetensors huggingface-hub chromadb sentence-transformers PyYAML python-dotenv rich tqdm numpy pandas pillow datasets requests beautifulsoup4 psutil

In [ ]:
# 2. Environment Settings & Auth
import os
from google.colab import userdata

# Optional: Set your Hugging Face Token in Colab Secrets as 'HF_TOKEN'
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("HF Token yüklendi.")
except:
    print("HF Token bulunamadı, unauthenticated devam ediliyor.")

In [ ]:
# 3. Clone Repository & Force Update
import os
import sys

REPO_URL = "https://github.com/halilibrahimavsar/mybabyai.git"
REPO_DIR = "mybabyai"

if os.path.exists(".git") or REPO_DIR in os.getcwd():
    print("Zaten repo dizinindeyiz, güncelleniyor...")
    !git fetch --all
    !git reset --hard origin/main || !git reset --hard origin/master
    !git pull
elif os.path.exists(REPO_DIR):
    print(f"'{REPO_DIR}' klasörü bulundu, içine giriliyor ve güncelleniyor...")
    %cd {REPO_DIR}
    !git fetch --all
    !git reset --hard origin/main || !git reset --hard origin/master
    !git pull
else:
    print("Repo bulunamadı, klonlanıyor...")
    !git clone {REPO_URL}
    %cd {REPO_DIR}

sys.path.insert(0, os.getcwd())
print(f"Çalışma dizini: {os.getcwd()}")

### 4. Initialize Model and Trainer

In [ ]:
# @title 🚀 CodeMind Eğitim Kontrol Paneli
# @markdown Bu hücreden eğitim türünü, model boyutunu ve başlangıç durumunu ayarlayabilirsiniz.

training_mode = "full" # @param ["full", "lora"]
model_size = "350M-MoE" # @param ["125M", "350M", "350M-MoE", "650M"]
load_from_checkpoint = False # @param {type:"boolean"}

print(f"🔹 Mod: {training_mode.upper()} | Boyut: {model_size} | Checkpoint: {"Evet" if load_from_checkpoint else "Hayır (Sıfırdan)"}")


In [ ]:
from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer
from src.utils.config import Config

config = Config()
model_manager = ModelManager(config)

if load_from_checkpoint:
    # Mevcut bir checkpointi (model_final.pt) yükler
    model_manager.load_model("codemind", allow_fresh_fallback=True)
else:
    # Belirlenen boyutta SIFIR bir model oluşturur
    model_manager.load_fresh_model(size=model_size)

trainer = LoRATrainer(model_manager, config)

### 5. Multi-Dataset Training

In [ ]:
# @title 5. Multi-Dataset Training
dataset_pool = [
    {
        "name": "English Chat (UltraChat)",
        "type": "huggingface",
        "dataset_key": "ultrachat_200k",
        "max_samples": 10000
    },
    {
        "name": "GPT-4 Instructions (SlimOrca)",
        "type": "huggingface",
        "dataset_key": "slim_orca",
        "max_samples": 10000
    },
    {
        "name": "Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "max_samples": 8000
    },
    {
        "name": "Turkish Alpaca",
        "type": "huggingface",
        "dataset_key": "turkish_alpaca",
        "max_samples": 5000
    },
    {
        "name": "Coding (CodeAlpaca)",
        "type": "huggingface",
        "dataset_key": "code_alpaca_20k",
        "max_samples": 5000
    }
]

metrics = trainer.train_from_pool(
    dataset_pool,
    training_type=training_mode,
    use_notebook_callback=True, 
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4 if training_mode == "full" else 1e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100
)

print(f"Training Complete {metrics}")